In [1]:
import sys, time
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

from src.config import load_config
from src.spark_session import get_spark
from src.evaluation.metrics import evaluate
from src.evaluation.stats import paired_bootstrap
from src.models.popularity import add_recent_counts, smoothed_ctr
from src.models.embeddings import user_profiles_dense, score_dense
from src.models.hybrid import pct_rank, zscore, weighted

cfg = load_config("small")
spark = get_spark(cfg)
OUT = cfg["paths"]["processed"]
RES = ROOT / "results" / "metrics"

WS = (0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0)
DELTA = 0.01
T_INICIO = time.time()

In [2]:
t0 = time.time()
imp = spark.read.parquet(str(OUT / "impressions_long"))
beh = spark.read.parquet(str(OUT / "behaviors"))
news_emb = spark.read.parquet(str(OUT / "news_emb_minilm")).cache()

users_val = (beh.filter("periodo = 'val'")
                .select("user_id", "history").dropDuplicates(["user_id"]).cache())

# Señal 1: CTR reciente (baseline principal)
p0 = imp.filter("periodo = 'train'").agg(F.avg("label")).first()[0]
base = (imp.filter(F.col("periodo").isin("train", "val"))
           .filter(F.col("time") >= F.to_timestamp(F.lit("2019-11-13 00:00:00")))
           .select("impression_id", "user_id", "time", "news_id", "label", "periodo"))
cands = (add_recent_counts(base, hours=(24,))
         .filter("periodo = 'val'")
         .withColumn("ctr", smoothed_ctr(24, 20, p0))
         .select("impression_id", "user_id", "news_id", "label", "ctr"))

# Señal 2: coseno de embeddings (historial completo)
profiles = user_profiles_dense(users_val, news_emb, k=None)

senales = (score_dense(cands, news_emb, profiles)
           .withColumn("ctr_rank", pct_rank("ctr"))
           .withColumn("cos_rank", pct_rank("cos"))
           .withColumn("ctr_z", zscore("ctr"))
           .withColumn("cos_z", zscore("cos"))
           .select("impression_id", "label", "ctr", "cos",
                   "ctr_rank", "cos_rank", "ctr_z", "cos_z")
           .cache())

print("Candidatas:", senales.count(), f"({time.time() - t0:.0f} s)")
senales.select(F.round(F.corr("ctr", "cos"), 3).alias("corr_ctr_cos")).show()

Candidatas: 1222429 (318 s)
+------------+
|corr_ctr_cos|
+------------+
|       0.036|
+------------+



In [3]:
def eval_score(df, expr, nombre):
    per_imp, res = evaluate(df.withColumn("score", expr))
    per_imp.unpersist()
    return {"modelo": nombre, **res}

filas = []
for norm in ("rank", "z"):
    for w in WS:
        t0 = time.time()
        nombre = f"hibrido_{norm}_w{w:.1f}"
        filas.append(eval_score(senales, weighted(f"cos_{norm}", f"ctr_{norm}", w), nombre))
        print(f"{nombre:18s} AUC = {filas[-1]['auc']:.4f} | nDCG@5 = {filas[-1]['ndcg5']:.4f} "
              f"({time.time() - t0:.0f} s)")

print(f"\nTiempo acumulado: {(time.time() - T_INICIO) / 60:.1f} min")

hibrido_rank_w0.0  AUC = 0.6475 | nDCG@5 = 0.2936 (9 s)
hibrido_rank_w0.2  AUC = 0.6650 | nDCG@5 = 0.3155 (4 s)
hibrido_rank_w0.4  AUC = 0.6834 | nDCG@5 = 0.3332 (2 s)
hibrido_rank_w0.5  AUC = 0.6819 | nDCG@5 = 0.3382 (1 s)
hibrido_rank_w0.6  AUC = 0.6739 | nDCG@5 = 0.3391 (1 s)
hibrido_rank_w0.8  AUC = 0.6546 | nDCG@5 = 0.3333 (1 s)
hibrido_rank_w1.0  AUC = 0.6352 | nDCG@5 = 0.3141 (2 s)
hibrido_z_w0.0     AUC = 0.6475 | nDCG@5 = 0.2936 (2 s)
hibrido_z_w0.2     AUC = 0.6712 | nDCG@5 = 0.3087 (1 s)
hibrido_z_w0.4     AUC = 0.6857 | nDCG@5 = 0.3303 (1 s)
hibrido_z_w0.5     AUC = 0.6831 | nDCG@5 = 0.3411 (2 s)
hibrido_z_w0.6     AUC = 0.6755 | nDCG@5 = 0.3469 (4 s)
hibrido_z_w0.8     AUC = 0.6597 | nDCG@5 = 0.3362 (3 s)
hibrido_z_w1.0     AUC = 0.6352 | nDCG@5 = 0.3141 (3 s)

Tiempo acumulado: 6.0 min


In [4]:
tabla = (pd.DataFrame(filas)[["modelo", "auc", "mrr", "ndcg5", "ndcg10", "impresiones"]]
         .sort_values("auc", ascending=False).reset_index(drop=True))
(tabla.assign(periodo="val", dataset=cfg["dataset"])
      .to_csv(RES / "06_hibrido_val.csv", index=False))

mejor = tabla.iloc[0]
print(f"Mejor híbrido en val: {mejor['modelo']}  AUC = {mejor['auc']:.4f}")
tabla.round(4)

Mejor híbrido en val: hibrido_z_w0.4  AUC = 0.6857


,modelo,auc,mrr,ndcg5,ndcg10,impresiones
0,hibrido_z_w0.4,0.6857,0.3060,0.3303,0.3939,30270
1,hibrido_rank_w0.4,0.6834,0.3103,0.3332,0.3959,30270
2,hibrido_z_w0.5,0.6831,0.3144,0.3411,0.4031,30270
3,hibrido_rank_w0.5,0.6819,0.3130,0.3382,0.4000,30270
4,hibrido_z_w0.6,0.6755,0.3194,0.3469,0.4077,30270
5,hibrido_rank_w0.6,0.6739,0.3121,0.3391,0.4000,30270
6,hibrido_z_w0.2,0.6712,0.2892,0.3087,0.3726,30270
7,hibrido_rank_w0.2,0.6650,0.2961,0.3155,0.3785,30270
8,hibrido_z_w0.8,0.6597,0.3103,0.3362,0.3960,30270
9,hibrido_rank_w0.8,0.6546,0.3070,0.3333,0.3927,30270


In [5]:
norm, w = mejor["modelo"].split("_")[1], float(mejor["modelo"].split("_w")[1])

per_h, _ = evaluate(senales.withColumn("score", weighted(f"cos_{norm}", f"ctr_{norm}", w)))
per_b, _ = evaluate(senales.withColumn("score", F.col("ctr")))
m = (per_h.select("impression_id", "auc", "ndcg5").toPandas()
     .merge(per_b.select("impression_id", "auc", "ndcg5").toPandas(),
            on="impression_id", suffixes=("_h", "_b")))

for met in ("auc", "ndcg5"):
    dif, ic, medias = paired_bootstrap(m[f"{met}_h"], m[f"{met}_b"])
    print(f"{met.upper():6s} diferencia = {dif:+.4f}  IC 95 % = [{ic[0]:+.4f}, {ic[1]:+.4f}]  "
          f"P(dif ≤ 0) = {np.mean(medias <= 0):.3f}  P(dif ≤ {DELTA}) = {np.mean(medias <= DELTA):.3f}")

AUC    diferencia = +0.0382  IC 95 % = [+0.0361, +0.0404]  P(dif ≤ 0) = 0.000  P(dif ≤ 0.01) = 0.000
NDCG5  diferencia = +0.0367  IC 95 % = [+0.0345, +0.0389]  P(dif ≤ 0) = 0.000  P(dif ≤ 0.01) = 0.000


In [6]:
print(f"Tiempo total: {(time.time() - T_INICIO) / 60:.1f} min")
spark.stop()

Tiempo total: 6.1 min
